# Ingestión del archivo `movie.csv`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo CSV usando `DataFrameReader` de Spark

In [0]:
from pyspark.sql.types import *
movie_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
])

movie_df = (spark.read 
    .schema(movie_schema)
    .option("header", True) 
    .csv(f"{bronze_folder_path}/{v_file_date}/movie.csv")
)


## 2. Seleccionar solo las columnas requeridas

In [0]:
# movies_selected_df = movie_df.select("movieId", "title", "budget", "popularity", "yearReleaseDate", "releaseDate", "revenue", "durationTime", "voteAverage", "voteCount")

# Otra forma
# movies_selected_df = movie_df.select(movie_df.movieId, movie_df.title, movie_df.budget, movie_df.popularity, movie_df.yearReleaseDate, movie_df.releaseDate, movie_df.revenue, movie_df.durationTime, movie_df.voteAverage, movie_df.voteCount)

# Otra forma
# movies_selected_df = movie_df.select(movie_df["movieId"], movie_df["title"], movie_df["budget"], movie_df["popularity"], movie_df["yearReleaseDate"], movie_df["releaseDate"], movie_df["revenue"], movie_df["durationTime"], movie_df["voteAverage"], movie_df["voteCount"])

# Otra forma
from pyspark.sql.functions import col
movies_selected_df = movie_df.select(col("movieId"), col("title"), col("budget"), col("popularity"), col("yearReleaseDate"), col("releaseDate"), col("revenue"), col("durationTime"), col("voteAverage"), col("voteCount"))

## 3. Cambiar el nombre de las columnas según lo requerido

In [0]:
movies_renamed_df = (movies_selected_df
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("yearReleaseDate", "year_release_date")
    .withColumnRenamed("releaseDate", "release_date")
    .withColumnRenamed("durationTime", "duration_time")
    .withColumnRenamed("voteAverage", "vote_average")
    .withColumnRenamed("voteCount", "vote_count"))

# Otra opción

# movies_renamed_df = (movies_selected_df
#    .withColumnRenamed({"movieId": "movie_id", "yearReleaseDate": "year_release_date", "releaseDate": "release_date",  "durationTime": "duration_time", "voteAverage": "vote_average", "voteCount": "vote_count"}))


## 4. Agregar la columna `ingestion_date` y `environment` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# movies_final_df = movies_renamed_df.withColumn("ingestion_date", current_timestamp())
movies_final_df = add_ingestion_date(movies_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))

## 5. Escribir datos en el datalake en formato `Delta`

In [0]:
merge_delta_lake( movies_final_df, "movie_silver", "movies", "tgt.movie_id = src.movie_id AND tgt.file_date = src.file_date", "file_date" )

In [0]:
%sql
SELECT * FROM movie_silver.movies;

movie_id,title,budget,popularity,year_release_date,release_date,revenue,duration_time,vote_average,vote_count,ingestion_date,enviroment,file_date
30979,The Horse Boy,2000000.0,0.465404,2009,2009-11-25,2600000.0,93,5.5,2,2026-09-14T18:28:18.067369Z,Production,2024-12-23
31005,Moonlight Mile,2.1E7,8.982335,2002,2002-09-09,1.001105E7,117,6.5,52,2026-09-14T18:28:18.067369Z,Production,2024-12-23
31007,Welcome to the Rileys,2000000.0,6.085879,2010,2010-10-29,2600000.0,110,6.4,139,2026-09-14T18:28:18.067369Z,Production,2024-12-23
31064,The Business of Strangers,2000000.0,1.369649,2001,2001-01-19,2600000.0,84,5.7,19,2026-09-14T18:28:18.067369Z,Production,2024-12-23
31117,Superbabies: Baby Geniuses 2,2000000.0,5.886228,2004,2004-08-27,2600000.0,88,1.9,35,2026-09-14T18:28:18.067369Z,Production,2024-12-23
31163,Chicken Tikka Masala,2000000.0,0.258413,2005,2005-04-22,2600000.0,90,3.5,5,2026-09-14T18:28:18.067369Z,Production,2024-12-23
31166,I Come with the Rain,1.8E7,8.266124,2009,2009-05-14,2.25E7,114,5.6,20,2026-09-14T18:28:18.067369Z,Production,2024-12-23
31174,Richard III,2000000.0,5.107235,1995,1995-12-29,2600000.0,104,6.9,49,2026-09-14T18:28:18.067369Z,Production,2024-12-23
31175,Soul Kitchen,5000000.0,5.461487,2009,2009-09-09,1.7872796E7,99,7.1,116,2026-09-14T18:28:18.067369Z,Production,2024-12-23
31203,Le petit Nicolas,2000000.0,20.362025,2009,2009-09-30,2600000.0,91,5.9,306,2026-09-14T18:28:18.067369Z,Production,2024-12-23
